In [1]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp

In [2]:
!pip install ecos
!pip install scs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 4.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# --- Hàm xây dựng Ma trận A (từ Ảnh 2) ---
def build_A_matrix(n_chargers: int, n_slots: int) -> np.ndarray:
    """
    Xây dựng ma trận ràng buộc A (slot-matrix constraint)
    Kích thước: (n_slots, n_chargers * n_slots)
    """
    # 1. Tạo ma trận đơn vị I_T (kích thước T x T)
    I_T = np.eye(n_slots)

    # 2. Lặp lại (tile) ma trận I_T C lần theo chiều ngang (1, C)
    A = np.tile(I_T, (1, n_chargers))

    return A

In [5]:
def buyer_best_response_cvx(v_i: np.ndarray,
                            p: np.ndarray,
                            A: np.ndarray,
                            q_i,
                            B_i: float) -> np.ndarray:
    """
    CVXPY solve of
        max  log(v_i · x)
        s.t. x >= 0 , (price + q_i)ᵀ x <= B_i
    """
    m = len(v_i) # Số lượng hàng hóa (n_goods)

    # Định nghĩa biến tối ưu
    x = cp.Variable(m, nonneg=True)

    # 1. Hàm mục tiêu (Objective)
    # Thêm B_i * bên ngoài log, theo Công thức 9
    utility = v_i @ x + 1e-12  # v_i · x (thêm 1e-12 để tránh log(0))
    objective = cp.Maximize(B_i * cp.log(utility))

    # 2. Ràng buộc ngân sách (Constraint)
    # Đây là "giá hiệu dụng" (effective price) mới
    effective_price = p + (A.T @ q_i)
    constraint = [effective_price @ x <= B_i]

    # 3. Giải bài toán
    prob = cp.Problem(objective, constraint)
    try:
        prob.solve(solver="ECOS", warm_start=True)
    except cp.SolverError:
        prob.solve(solver="SCS")

    return x.value

In [6]:
from google.colab import drive

# BƯỚC 1: Kết nối Google Drive (chỉ cần chạy 1 lần)
drive.mount('/content/drive', force_remount=True)

# BƯỚC 2: Cập nhật đường dẫn FACTOR_DIR
# Giả sử bạn đặt thư mục "valuation_out_23h_sigmoid"
# ngay trong "My Drive" (Drive của tôi)
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_50u_60i"

# --- Code của bạn bắt đầu từ đây ---
try:
    U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
    P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
    valuations = U @ P.T
    print(f"✓ Đã tải xong factors. Ma trận Valuations có shape: {valuations.shape}")

except Exception as e:
    print(f"⚠️  Lỗi khi tải file: {e}")
    print("Sử dụng ma trận valuations ngẫu nhiên...")
    valuations = np.random.rand(120, 12) + 0.5         # 120 buyers, 12 goods

# --- Phần còn lại của code giữ nguyên ---
n_buyers, n_goods = valuations.shape
budgets = np.full(n_buyers, 1.0)
energy  = np.ones(n_buyers)
p0      = np.ones(n_goods)
n_slots = 4
q0 = np.ones((n_buyers, n_slots))
b_constraint = np.ones((n_buyers, n_slots))

obj_tol = 1e-3
num_iters = 10 ** 100
log_freq  = 100
X_tol   = 1e-5
seed = 2001
timE = 7200

print(f"\nThiết lập hoàn tất. (n_buyers={n_buyers}, n_goods={n_goods})")

Mounted at /content/drive
✓ Đã tải xong factors. Ma trận Valuations có shape: (50, 60)

Thiết lập hoàn tất. (n_buyers=50, n_goods=60)


In [21]:
# --- BẠN BẮT ĐẦU TỪ ĐÂY ---

# 1. Giả sử bạn đã tải 'valuations'
# valuations = np.load(...)

# 2. Lấy n_goods từ valuations
n_buyers, n_goods = valuations.shape

# 3. Định nghĩa n_slots (từ ảnh của bạn)
n_slots = 10

# 4. Tính toán n_chargers
# (Kiểm tra xem n_goods có chia hết cho n_slots không)
if n_goods % n_slots != 0:
    print(f"⚠️ Cảnh báo: n_goods ({n_goods}) không chia hết cho n_slots ({n_slots})")

n_chargers = n_goods // n_slots

# 5. --- TẠO MA TRẬN A ĐỂ TRUYỀN VÀO HÀM ---
A_matrix = build_A_matrix(n_chargers, n_slots)

print(f"Đã tạo Ma trận A với shape: {A_matrix.shape}")

Đã tạo Ma trận A với shape: (10, 60)


In [8]:
def plot_constraint_violation(
    hist_base,
    hist_full,
    hist_sgd=None,
    sgd_log_freq=1,
    full_log_freq=1,
    title="Constraint Satisfaction Convergence",
    y_label=r'Mean Residual $\frac{1}{T} \sum (Ax - b)$',
    xlim=None
):
    """
    Vẽ biểu đồ so sánh mức độ vi phạm ràng buộc giữa các thuật toán.

    Tham số:
    - hist_base: List lịch sử của Baseline (Full Grad, q=0)
    - hist_full: List lịch sử của Proposed Full Gradient (q>0)
    - hist_sgd:  (Tùy chọn) List lịch sử của Proposed SGD. Nếu None sẽ không vẽ.
    - sgd_log_freq: Tần suất ghi log của SGD (để scale trục X đúng)
    - full_log_freq: Tần suất ghi log của Full Grad
    - xlim: (Tùy chọn) Giới hạn trục X (ví dụ: 1000) để zoom vào đoạn đầu.
    """

    plt.figure(figsize=(10, 6))

    # 1. Tạo trục X cho Full Gradient (Baseline & Proposed Full)
    x_full = np.arange(len(hist_base)) * full_log_freq

    # Vẽ Baseline (Màu xám, nét đứt)
    plt.plot(x_full, hist_base,
             color='gray', linestyle='--', linewidth=2, alpha=0.8,
             label='Baseline (Eisenberg-Gale)')

    # Vẽ Proposed Full (Màu xanh lá, nét liền)
    plt.plot(x_full, hist_full,
             color='#008000', linestyle='-', linewidth=2.5, alpha=0.9,
             label='Proposed (Full Gradient)')

    # 2. Vẽ SGD (Nếu có)
    if hist_sgd is not None:
        x_sgd = np.arange(len(hist_sgd)) * sgd_log_freq
        plt.plot(x_sgd, hist_sgd,
                 color='darkorange', linestyle='-', linewidth=1.5, alpha=0.7,
                 label='Proposed (SGD)')

    # 3. Trang trí biểu đồ
    # Đường an toàn y=0
    plt.axhline(0, color='black', linewidth=1.5, linestyle='-', alpha=0.4)
    plt.text(0, 0.05, ' Overloaded (Positive)', color='red', fontsize=9, verticalalignment='bottom', fontweight='bold')
    plt.text(0, -0.05, ' Safe Zone (Negative)', color='blue', fontsize=9, verticalalignment='top', fontweight='bold')

    # Chú thích "Infeasible Plateau" cho Baseline
    if len(hist_base) > 0:
        end_val = hist_base[-1]
        mid_x = len(hist_base) // 2 * full_log_freq
        # Chỉ vẽ mũi tên nếu giá trị dương (đang vi phạm)
        if end_val > 0.1:
            plt.annotate('Infeasible Plateau',
                         xy=(mid_x, end_val),
                         xytext=(mid_x + 50, end_val + 0.5),
                         arrowprops=dict(facecolor='black', shrink=0.05),
                         fontsize=10)

    plt.xlabel('Iterations', fontsize=12)
    plt.ylabel(y_label, fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(fontsize=11, loc='upper right', frameon=True, shadow=True)

    # Zoom trục X nếu cần
    if xlim:
        plt.xlim(0, xlim)

    plt.tight_layout()
    plt.show()

In [9]:
import time
import numpy as np
np.seterr(all='ignore') # Tắt cảnh báo log(0)

import time
import numpy as np

def one_sample_sgd(
    A: np.ndarray,
    b: np.ndarray,
    supply_s: np.ndarray,
    q0: np.ndarray,
    valuations: np.ndarray,
    budgets: np.ndarray,
    p0: np.ndarray,
    lr_p=0.05, lr_q=0.05,
    num_iters=500,      # SGD cần nhiều bước hơn
    seed=0,
    log_freq=10,         # Log thường xuyên hơn
    update_q=True        # Tham số bật/tắt Baseline
):
    rng = np.random.default_rng(seed)
    n, m = valuations.shape
    T = A.shape[0]

    p = p0.astype(float).copy()
    q = q0.astype(float).copy()

    if not update_q:
        q = np.zeros_like(q)

    # Warm-start X (để tính D ban đầu)
    X = np.zeros((n, m))
    for j in range(n):
        # Lưu ý: Hàm solver phải xử lý được effective_price = p + A.T @ q
        X[j] = buyer_best_response_cvx(valuations[j], p, A, q, budgets[j])

    # Tổng cầu ban đầu
    D = X.sum(axis=0)

    metric_hist = [] # Lưu Mean Residual

    for t in range(1, num_iters + 1):
        # 1. Chọn user ngẫu nhiên & Cập nhật D
        i = int(rng.integers(n))
        D -= X[i]

        x_i_new = buyer_best_response_cvx(valuations[i], p, A, q, budgets[i])
        if x_i_new is None: x_i_new = X[i]

        X[i] = x_i_new
        D += x_i_new

        # 2. Tính Load hiện tại (Incremental)
        current_load = A @ D

        # 3. Cập nhật biến (Stochastic Gradient)
        # Gradient p: Demand - Supply
        g_p = D - supply_s
        # Gradient q: Load - Capacity (b)
        g_q_update = current_load - b

        alpha = lr_p / np.sqrt(t)
        beta  = lr_q / np.sqrt(t)

        p = np.maximum(p + alpha * g_p, 0.0)

        if update_q:
            q = np.maximum(q + beta * g_q_update, 0.0)

        # 4. Log Metric (Mean Residual)
        if t % log_freq == 0:
            # Metric: Mean(Ax - b)
            # Dương = Quá tải, Âm = Dư thừa
            residual = current_load - b
            mean_residual = np.mean(residual)
            metric_hist.append(mean_residual)

    return metric_hist

In [31]:
import numpy as np
import cvxpy as cp
import time
np.seterr(all='ignore') # Tắt cảnh báo log(0)

def full_grad_descent(
    A: np.ndarray,
    b: np.ndarray,
    supply_s: np.ndarray,
    q0: np.ndarray,
    valuations: np.ndarray,
    budgets: np.ndarray,
    p0: np.ndarray,
    lr_p=0.05, lr_q=0.05,
    num_iters=500,
    log_freq=1,
    update_q=True
):
    n, m = valuations.shape
    T = A.shape[0]

    p = p0.astype(float).copy()
    q = q0.astype(float).copy()

    if not update_q:
        q = np.zeros_like(q)

    # Đổi tên list này thành metric_hist cho tổng quát
    violation_hist = []

    t0 = time.perf_counter()

    for t in range(1, num_iters + 1):
        # 1. Best Response
        X_list = []
        for j in range(n):
            # Lưu ý: effective_price = p + A.T @ q
            # (Bạn cần đảm bảo hàm buyer_best_response_cvx xử lý đúng input này)
            val = buyer_best_response_cvx(valuations[j], p, A, q, budgets[j])
            X_list.append(val)
        X = np.array(X_list)

        # 2. Tính toán Metric (Mean Residual) - THEO YÊU CẦU CỦA BẠN
        total_demand_goods = np.sum(X, axis=0)
        current_load = A @ total_demand_goods

        # 1. Lọc lấy phần dương (chỉ lấy phần vi phạm)
        positive_violation = np.maximum(0, current_load - b)

        # 2. Tính Max Norm (Theo đúng công thức trong ảnh bài báo)
        # Nghĩa là: "Slot nào đang bị vi phạm nặng nhất?"
        violation_metric = np.max(positive_violation)

        # (Hoặc nếu bạn muốn xem trung bình các vi phạm):
        # violation_metric = np.mean(positive_violation)

        violation_hist.append(violation_metric)
        # -----------------------

        # 3. Tính Gradient
        g_p = supply_s - total_demand_goods
        g_q = b - current_load # Gradient hướng xuống cho bài toán đối ngẫu (hoặc ngược lại tùy dấu)
        # Để thống nhất với logic update q = q + lr * (Ax - b):
        # Gradient dương khi vi phạm -> q tăng.
        # Nên dùng g_q_update = current_load - b
        g_q_update = current_load - b

        # 4. Cập nhật biến
        alpha = lr_p / np.sqrt(t)
        beta  = lr_q / np.sqrt(t)

        p = np.maximum(p - alpha * g_p, 0.0)

        if update_q:
            q = np.maximum(q + beta * g_q_update, 0.0) # Cộng gradient hướng vi phạm

        # Log
        if t % 10 == 0:
            print(f"Iter {t}: Violation = {violation_metric:.4f}")

    return violation_hist

In [ ]:
# --- 2. CHẠY CÁC THUẬT TOÁN ---
N_USERS, M_GOODS = valuations.shape

T_SLOTS = 10  # Giả định số slot/ngày (hoặc số ngày gộp) là 4
A = A_matrix
seed_list = [1,2,3,4,5,6,7,8,10,11,12,13,14,15]
res_list = [0] * len(seed_list)
for i in seed_list:
    np.random.seed(i)
    q0 = np.zeros(T_SLOTS) # Vector độ dài 10
    # 1. Tạo tham số ngẫu nhiên
    budget = np.random.uniform(5.0, 20.0, N_USERS)
    supply_s = np.random.uniform(0.5, 3.5, M_GOODS)

    # 2. Tạo Capacity (Sửa lỗi Capacity quá nhỏ)

    capacity_b = np.random.uniform(1, 4, T_SLOTS)
    q0 = np.zeros(T_SLOTS) # Vector độ dài 10

    print("\n1. Running Baseline (q=0)...")
    hist_base = full_grad_descent(A, capacity_b, supply_s, q0, valuations, budgets, p0, lr_p=0.1, lr_q=0.0, num_iters=300, update_q=False)

    print("2. Running Proposed Full (Update q)...")
    hist_full = full_grad_descent(A, capacity_b, supply_s, q0, valuations, budgets, p0, lr_p=0.1, lr_q=0.1, num_iters=300, update_q=True)

    res_list[i] = [hist_base, hist_full]


1. Running Baseline (q=0)...
Iter 10: Violation = 6.2719
Iter 20: Violation = 5.2320
Iter 30: Violation = 5.0094
Iter 40: Violation = 5.0251
Iter 50: Violation = 5.0387
Iter 60: Violation = 5.0507
Iter 70: Violation = 5.0575
Iter 80: Violation = 5.0600
Iter 90: Violation = 5.0635
Iter 100: Violation = 5.0662
Iter 110: Violation = 5.0687
Iter 120: Violation = 5.0705
Iter 130: Violation = 5.0718
Iter 140: Violation = 5.0749
Iter 150: Violation = 5.0773
Iter 160: Violation = 5.0796
Iter 170: Violation = 5.0813
Iter 180: Violation = 5.0834
Iter 190: Violation = 5.0850
Iter 200: Violation = 5.0868
Iter 210: Violation = 5.0883
Iter 220: Violation = 5.0892
Iter 230: Violation = 5.0913
Iter 240: Violation = 5.0925
Iter 250: Violation = 5.0940
Iter 260: Violation = 5.0951
Iter 270: Violation = 5.0965
Iter 280: Violation = 5.0976
Iter 290: Violation = 5.0989
Iter 300: Violation = 5.0998
2. Running Proposed Full (Update q)...
Iter 10: Violation = 9.8732
Iter 20: Violation = 6.6118
Iter 30: Viola

In [ ]:
for i in range(len(seed_list)):
    hist_base = res_list[i][0]
    hist_full = res_list[i][1]
    print("\nĐang vẽ biểu đồ...")
    plot_constraint_violation(hist_base, hist_full)